Underneath, in this notebook, you may find the prepared, e.t. filled with proper math and logic dataset, which performs greatly (>top 3%) even with untuned Deep Learning Model, which for this task is not suitable (obviously it do requires ML model, due to the highly dispersed low number of elements data datasets).

In [33]:
#Imports
import tensorflow as tf
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from google.colab.data_table import DataTable # Comment this line if not on Collab
pd.set_option('display.max_columns', None) # Comment this line if not on Collab
DataTable.max_columns = 30 # Comment this line if not on Collab

In [34]:
# Getting the data from github
!wget -O test.csv https://raw.githubusercontent.com/LozinskiMatthew/Titanic-DL-Kaggle_Competition/main/test.csv
!wget -O train.csv https://raw.githubusercontent.com/LozinskiMatthew/Titanic-DL-Kaggle_Competition/main/train.csv
test_data = pd.read_csv('../content/test.csv')
train_data = pd.read_csv('../content/train.csv')

--2025-02-04 17:37:06--  https://raw.githubusercontent.com/LozinskiMatthew/Titanic-DL-Kaggle_Competition/main/test.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 28210 (28K) [text/plain]
Saving to: ‘test.csv’

test.csv            100%[===================>]  27.55K  --.-KB/s    in 0.002s  

2025-02-04 17:37:06 (14.8 MB/s) - ‘test.csv’ saved [28210/28210]

--2025-02-04 17:37:06--  https://raw.githubusercontent.com/LozinskiMatthew/Titanic-DL-Kaggle_Competition/main/train.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 60302 

In [35]:
# Data cleansed, additionally PCA was adjusted,
finding_patterns = pd.concat([train_data, test_data], axis=0) # Name finding patterns was made, in the bigger verions, where I did extensive data exploration
train_data['Age'] = train_data['Age'].fillna(value=finding_patterns['Age'].mean())
train_data['Fare'] = train_data['Fare'].fillna(value=finding_patterns['Fare'].mean())
test_data['Age'] = test_data['Age'].fillna(value=finding_patterns['Age'].mean())
test_data['Fare'] = test_data['Fare'].fillna(value=finding_patterns['Fare'].mean())
train_data['Embarked'] = train_data['Embarked'].fillna(value='S') # There are two Nans, thus I wont create a new column, it's not worth it to make more dimensions (+ S dominates)
train_data['LetterInCabin'] = train_data['Cabin'].str[0]
test_data['LetterInCabin'] = test_data['Cabin'].str[0]
train_data = train_data.drop(['Cabin'], axis=1)
test_data = test_data.drop(['Cabin'], axis=1)
train_data['Name'].str.extract('([A-Za-z]+)\.', expand=True)
train_data['Name'] = train_data['Name'].str.extract('([A-Za-z]+)\.', expand=True)
train_data.loc[train_data['Age'] < 15, 'Name'] = 'Kid' # The crew considered them to be children
train_data['LetterInCabin'] = train_data['LetterInCabin'].fillna(value='N')
test_data['Name'] = test_data['Name'].str.extract('([A-Za-z]+)\.', expand=True)
test_data.loc[test_data['Age'] < 15, 'Name'] = 'Kid'
test_data['LetterInCabin'] = test_data['LetterInCabin'].fillna(value='N')

In [36]:
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,LetterInCabin
0,1,0,3,Mr,male,22.0,1,0,A/5 21171,7.2500,S,N
1,2,1,1,Mrs,female,38.0,1,0,PC 17599,71.2833,C,C
2,3,1,3,Miss,female,26.0,0,0,STON/O2. 3101282,7.9250,S,N
3,4,1,1,Mrs,female,35.0,1,0,113803,53.1000,S,C
4,5,0,3,Mr,male,35.0,0,0,373450,8.0500,S,N


In [37]:
# Normalization
scaler = MinMaxScaler()

ready_train_dataset_scaled = scaler.fit_transform(train_data.drop(['Sex', 'Survived', 'Pclass', 'Ticket', 'Embarked', 'LetterInCabin', 'Name'], axis=1))
ready_train_dataset_scaled_df = pd.DataFrame(data=ready_train_dataset_scaled, columns=train_data.drop(['Sex', 'Survived', 'Pclass', 'Ticket', 'Embarked', 'LetterInCabin', 'Name'], axis=1).columns)
train_data = pd.concat([train_data.drop(['PassengerId', 'Age', 'SibSp', 'Parch', 'Fare'], axis=1), ready_train_dataset_scaled_df], axis=1)

ready_test_dataset_scaled = scaler.transform(test_data.drop(['Sex', 'Pclass', 'Ticket', 'Embarked', 'LetterInCabin', 'Name'], axis=1))
ready_test_dataset_scaled_df = pd.DataFrame(data=ready_test_dataset_scaled, columns=test_data.drop(['Sex', 'Pclass', 'Ticket', 'Embarked', 'LetterInCabin', 'Name'], axis=1).columns)
test_data = pd.concat([test_data.drop(['PassengerId', 'Age', 'SibSp', 'Parch', 'Fare'], axis=1), ready_test_dataset_scaled_df], axis=1)


In [38]:
train_data['LetterInCabin'].value_counts()

,count
LetterInCabin,
N,687
C,59
B,47
D,33
E,32
A,15
F,13
G,4
T,1


In [39]:
# Standardization will be better if you were to decide to work with ML models, which will perform better

'''
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

ready_train_dataset_scaled = scaler.fit_transform(train_data.drop(['Sex', 'Survived', 'Pclass', 'Ticket', 'Embarked', 'LetterInCabin', 'Name'], axis=1))
ready_train_dataset_scaled_df = pd.DataFrame(data=ready_train_dataset_scaled, columns=train_data.drop(['Sex', 'Survived', 'Pclass', 'Ticket', 'Embarked', 'LetterInCabin', 'Name'], axis=1).columns)
train_data = pd.concat([train_data.drop(['PassengerId', 'Age', 'SibSp', 'Parch', 'Fare'], axis=1), ready_train_dataset_scaled_df], axis=1)

ready_test_dataset_scaled = scaler.fit_transform(test_data.drop(['Sex', 'Pclass', 'Ticket', 'Embarked', 'LetterInCabin', 'Name'], axis=1))
ready_test_dataset_scaled_df = pd.DataFrame(data=ready_test_dataset_scaled, columns=test_data.drop(['Sex', 'Pclass', 'Ticket', 'Embarked', 'LetterInCabin', 'Name'], axis=1).columns)
test_data = pd.concat([test_data.drop(['PassengerId', 'Age', 'SibSp', 'Parch', 'Fare'], axis=1), ready_test_dataset_scaled_df], axis=1)

'''

"\nfrom sklearn.preprocessing import StandardScaler\n\nscaler = StandardScaler()\n\nready_train_dataset_scaled = scaler.fit_transform(train_data.drop(['Sex', 'Survived', 'Pclass', 'Ticket', 'Embarked', 'LetterInCabin', 'Name'], axis=1))\nready_train_dataset_scaled_df = pd.DataFrame(data=ready_train_dataset_scaled, columns=train_data.drop(['Sex', 'Survived', 'Pclass', 'Ticket', 'Embarked', 'LetterInCabin', 'Name'], axis=1).columns)\ntrain_data = pd.concat([train_data.drop(['PassengerId', 'Age', 'SibSp', 'Parch', 'Fare'], axis=1), ready_train_dataset_scaled_df], axis=1)\n\nready_test_dataset_scaled = scaler.fit_transform(test_data.drop(['Sex', 'Pclass', 'Ticket', 'Embarked', 'LetterInCabin', 'Name'], axis=1))\nready_test_dataset_scaled_df = pd.DataFrame(data=ready_test_dataset_scaled, columns=test_data.drop(['Sex', 'Pclass', 'Ticket', 'Embarked', 'LetterInCabin', 'Name'], axis=1).columns)\ntest_data = pd.concat([test_data.drop(['PassengerId', 'Age', 'SibSp', 'Parch', 'Fare'], axis=1), re

In [40]:
# OneHotEncoding
ohe = OneHotEncoder(sparse_output=False, handle_unknown='infrequent_if_exist')

embarked_encoded_train = ohe.fit_transform(train_data[['Embarked']])
embarked_train_df = pd.DataFrame(data=embarked_encoded_train, columns=ohe.categories_)
embarked_encoded_test = ohe.transform(test_data[['Embarked']])
embarked_test_df = pd.DataFrame(data=embarked_encoded_test, columns=ohe.categories_)

categories=[['male', 'female']]
ore_sex = OrdinalEncoder(categories=categories)
sex_encoded_train = ore_sex.fit_transform(train_data[['Sex']])
sex_train_df = pd.DataFrame(data=sex_encoded_train, columns=['Sex'])
sex_encoded_test = ore_sex.transform(test_data[['Sex']])
sex_test_df = pd.DataFrame(data=sex_encoded_test, columns=['Sex'])

categories = [['T', 'H', 'G', 'F', 'N', 'E', 'D', 'C', 'B', 'A']]
ore_cabin = OrdinalEncoder(categories=categories)
letter_encoded_train = ore_cabin.fit_transform(train_data[['LetterInCabin']])
letter_train_df = pd.DataFrame(data=letter_encoded_train, columns=['LetterInCabin']) # I put N before E due to the previous analysis and common sense i.e. analysis + it cannot be in the middle directly, cause the ones with lower feasibilities, were the ones that couldnt buy better tickets, and obviously we must have more data on rich people
letter_encoded_test = ore_cabin.transform(test_data[['LetterInCabin']])
letter_test_df = pd.DataFrame(data=letter_encoded_test, columns=['LetterInCabin'])

ticket_encoded_train = ohe.fit_transform(train_data[['Ticket']])
ticket_train_df = pd.DataFrame(data=ticket_encoded_train, columns=ohe.categories_)
ticket_encoded_test = ohe.transform(test_data[['Ticket']])
ticket_test_df = pd.DataFrame(data=ticket_encoded_test, columns=ohe.categories_)

name_encoded_train = ohe.fit_transform(train_data[['Name']])
name_train_df = pd.DataFrame(data=name_encoded_train, columns=ohe.categories_)
name_encoded_test = ohe.transform(test_data[['Name']])
name_test_df = pd.DataFrame(data=name_encoded_test, columns=ohe.categories_)

final_train_data = pd.concat([train_data.drop(['Sex', 'Ticket', 'Embarked', 'LetterInCabin', 'Name'], axis=1), embarked_train_df, sex_train_df, letter_train_df, ticket_train_df, name_train_df], axis=1)
final_test_data = pd.concat([test_data.drop(['Sex', 'Ticket', 'Embarked', 'LetterInCabin', 'Name'], axis=1), embarked_test_df, sex_test_df, letter_test_df, ticket_test_df, name_test_df], axis=1)
final_test_data = final_test_data.reindex(columns=final_train_data.columns, fill_value=0.0)
final_test_data.drop(columns=['Survived'], inplace=True)

In [41]:
# Float64 convergence, you may need it in some cases, and it does not hurt the algorithm.
final_train_data['SibSp'] =  final_train_data['SibSp'].astype('float64')
final_train_data['Parch'] =  final_train_data['Parch'].astype('float64')
final_train_data['Survived'] =  final_train_data['Survived'].astype('float64')

final_test_data['SibSp'] =  final_test_data['SibSp'].astype('float64')
final_test_data['Parch'] =  final_test_data['Parch'].astype('float64')
final_test_data['PassengerId'] =  final_test_data['PassengerId'].astype('float64')

In [42]:
# Creating final datasets to work with.
final_train_data_labels = final_train_data.drop(columns=['Survived'])
final_train_data_target = final_train_data['Survived']
train_labels_np = final_train_data_labels.to_numpy()
train_target_np = final_train_data_target.to_numpy()
test_np = final_test_data.to_numpy()

In [43]:
#For training your models, experimenting with your models, you should use validation dataset, e.t. uncomment below, and use it for fit, and add proper callbacks
'''
divisor = np.round(float(len(final_train_data)) * 85 / 100)
divisor = int(divisor)
train_np_labels = final_train_data_labels[:divisor].to_numpy()
train_np_labels = final_train_data_target[:divisor].to_numpy()
valid_np_labels = final_train_data_labels[divisor:].to_numpy()
valid_np_target = final_train_data_target[divisor:].to_numpy()
'''

'\ndivisor = np.round(float(len(final_train_data)) * 85 / 100)\ndivisor = int(divisor)\ntrain_np_labels = final_train_data_labels[:divisor].to_numpy()\ntrain_np_labels = final_train_data_target[:divisor].to_numpy()\nvalid_np_labels = final_train_data_labels[divisor:].to_numpy()\nvalid_np_target = final_train_data_target[divisor:].to_numpy()\n'

In [44]:
# Func with Callbacks, were you to choose validation dataset, remember to uncomment these lines, tip. you can do it simultaneously via holding alt key, and pointing directions then ctrl+/
class CallbackCreator(tf.keras.callbacks.Callback):
  def on_epoch_end(self, epoch, logs=None):
    print(f"Epoch {epoch+1}:")
    print(f" - Loss: {logs['loss']}")
    print(f" - Accuracy: {logs['accuracy']}")
    print(f" - Precision: {logs['precision']}")
    # print(f" - Validation Loss: {logs['val_loss']}")
    # print(f" - Validation mae: {logs['val_accuracy']}")
    # print(f" = Validation Precision: {logs['val_precision']}")

That is the place for your model, were you to choose to go with DL, just write your model below, but in ML case, you should uncomment the standardization, and comment the normalization cell.

And beneath, lies a way to add to your model, final results, e.t. survived or not, just replace model_0 with  your current model name.

In [45]:
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='accuracy', patience=7, restore_best_weights=True)

In [52]:
# Your model_0, random model that scores in top 3% with DL

model_0 = tf.keras.Sequential([
    tf.keras.layers.Dense(80, activation='relu'),
    tf.keras.layers.Dense(60, activation='relu'),
    tf.keras.layers.Dense(40, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

loss = tf.keras.losses.BinaryFocalCrossentropy()
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)
model_0.compile(loss=loss, optimizer=optimizer, metrics=['accuracy', 'precision', 'recall'])

model_0.fit(x=train_labels_np, y=train_target_np, epochs=120, verbose=1, callbacks=[CallbackCreator(), early_stopping])


Epoch 1/120
24/28 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.6405 - loss: 0.1658 - precision: 0.7320 - recall: 0.0704Epoch 1:
 - Loss: 0.16293737292289734
 - Accuracy: 0.6285073161125183
 - Precision: 0.7894737124443054
28/28 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.6384 - loss: 0.1653 - precision: 0.7415 - recall: 0.0660
Epoch 2/120
21/28 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6356 - loss: 0.1562 - precision: 0.9048 - recall: 0.0411Epoch 2:
 - Loss: 0.1542862355709076
 - Accuracy: 0.6487092971801758
 - Precision: 1.0
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6377 - loss: 0.1558 - precision: 0.9310 - recall: 0.0514
Epoch 3/120
26/28 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.6805 - loss: 0.1507 - precision: 0.9655 - recall: 0.2258Epoch 3:
 - Loss: 0.1465526670217514
 - Accuracy: 0.7227833867073059
 - Precision: 0.9439252614974976
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.6848 - loss: 0.1502 - precision: 0.9635 - recall: 0.2330
Epoch 4/1

In [53]:
test_data_v2 = pd.read_csv('../content/test.csv')
final_result = pd.DataFrame(data=test_data_v2['PassengerId'], columns=['PassengerId'])
test_length = len(final_test_data)
for i in range(test_length):
  input_data = test_np[i].reshape(1, -1)
  prediction = int(np.round(model_0.predict(input_data, verbose=1))) # You may set verbose to 0, lest you do not want to see the progress
  final_result.loc[i, 'Survived'] = prediction
  print(prediction)
final_result['Survived'] = final_result['Survived'].astype(int)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step
0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1


<ipython-input-53-e8a844c278e3>:6: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  prediction = int(np.round(model_0.predict(input_data, verbose=1))) # You may set verbose to 0, lest you do not want to see the progress
<ipython-input-53-e8a844c278e3>:6: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  prediction = int(np.round(model_0.predict(input_data, verbose=1))) # You may set verbose to 0, lest you do not want to see the progress


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
0


<ipython-input-53-e8a844c278e3>:6: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  prediction = int(np.round(model_0.predict(input_data, verbose=1))) # You may set verbose to 0, lest you do not want to see the progress
<ipython-input-53-e8a844c278e3>:6: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  prediction = int(np.round(model_0.predict(input_data, verbose=1))) # You may set verbose to 0, lest you do not want to see the progress


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step


...

<ipython-input-53-e8a844c278e3>:6: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  prediction = int(np.round(model_0.predict(input_data, verbose=1))) # You may set verbose to 0, lest you do not want to see the progress


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
0


<ipython-input-53-e8a844c278e3>:6: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  prediction = int(np.round(model_0.predict(input_data, verbose=1))) # You may set verbose to 0, lest you do not want to see the progress
<ipython-input-53-e8a844c278e3>:6: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  prediction = int(np.round(model_0.predict(input_data, verbose=1))) # You may set verbose to 0, lest you do not want to see the progress


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1


<ipython-input-53-e8a844c278e3>:6: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  prediction = int(np.round(model_0.predict(input_data, verbose=1))) # You may set verbose to 0, lest you do not want to see the progress


At the end you can see how to save it to the file.

In [55]:
final_result.to_csv('my_submission.csv', index=False)

You may submit your results, via copying the final_result.csv to the Kaggle Compatition site.